In [9]:
"""
Script pour convertir tous les fichiers XML en TXT en retirant les balises.
Les fichiers TXT sont sauvegardés dans un dossier 'text_dir/' avec le même nom.
"""

from pathlib import Path
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

def clean_text(text: str) -> str:
    """
    Retire toutes les balises XML/HTML du texte.
    """
    try:
        # Pour les fichiers de taille raisonnable, utiliser BeautifulSoup
        if len(text) < 1_000_000:  # < 1MB
            soup = BeautifulSoup(text, 'lxml')
            cleaned = soup.get_text(separator=' ', strip=True)
        else:
            # Pour les gros fichiers, utiliser regex (plus rapide)
            cleaned = re.sub(r'<[^>]+>', ' ', text)
    except Exception as e:
        print(f"  Erreur BeautifulSoup, utilisation de regex: {e}")
        cleaned = re.sub(r'<[^>]+>', ' ', text)
    
    # Nettoyer les entités HTML
    cleaned = re.sub(r'&[a-zA-Z0-9#]+;', ' ', cleaned)
    
    # Nettoyer les espaces multiples
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    return cleaned


def convert_xml_to_txt(input_dir: str = "xml_dir", output_dir: str = "text_dir"):
    """
    Convertit tous les fichiers XML en TXT en retirant les balises.
    
    Args:
        input_dir: Dossier contenant les fichiers XML
        output_dir: Dossier de sortie pour les fichiers TXT
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    # Créer le dossier de sortie
    output_path.mkdir(exist_ok=True)
    
    # Lister tous les fichiers XML
    xml_files = list(input_path.glob("*.xml"))
    
    if not xml_files:
        print(f"❌ Aucun fichier XML trouvé dans '{input_dir}'")
        return
    
    print(f"📁 {len(xml_files)} fichiers XML trouvés")
    print(f"🔄 Conversion en cours...\n")
    
    success_count = 0
    error_count = 0
    
    # Traiter chaque fichier
    for xml_file in tqdm(xml_files, desc="Conversion"):
        try:
            # Lire le fichier XML
            with open(xml_file, 'r', encoding='utf-8') as f:
                xml_content = f.read()
            
            # Nettoyer le texte
            cleaned_text = clean_text(xml_content)
            
            # Créer le nom du fichier TXT
            txt_file = output_path / f"{xml_file.stem}.txt"
            
            # Sauvegarder le fichier TXT
            with open(txt_file, 'w', encoding='utf-8') as f:
                f.write(cleaned_text)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n❌ Erreur sur {xml_file.name}: {e}")
            error_count += 1
    
    print(f"\n✅ Conversion terminée!")
    print(f"   - Réussis: {success_count}")
    print(f"   - Erreurs: {error_count}")
    print(f"   - Fichiers dans: {output_path.absolute()}")


INPUT_DIR = "xml_dir"
OUTPUT_DIR = "text_dir"
    
convert_xml_to_txt(INPUT_DIR, OUTPUT_DIR)

📁 2634 fichiers XML trouvés
🔄 Conversion en cours...



Conversion: 100%|██████████| 2634/2634 [01:10<00:00, 37.36it/s]


✅ Conversion terminée!
   - Réussis: 2634
   - Erreurs: 0
   - Fichiers dans: c:\Users\alaa-\Documents\tekno_ai\rag\text_dir


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5050 Laptop GPU
13.0


In [ ]:
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os
import datetime


load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")


# 1. Charger le DataFrame filtré
df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)


#clean & prepare metadata
df = df.set_index("hash_id")
df["language"] = "english"
df["data source"] = "eur-lex"
df["date of migration"] = str(datetime.datetime.now().date())
df["Date of document"] = (
    df["Date of document"]
    .astype(str)
    .str.replace(r"[:;].*$", "", regex=True)
    .str.strip()
)
df.columns = df.columns.str.replace(":", "")
df.to_csv("b_hashed_list.csv")

In [2]:
import pandas as pd
df = pd.read_csv(
    "b_hashed_list.csv",
    index_col="hash_id",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)


In [3]:
from llama_index.core import SimpleDirectoryReader, StorageContext, Settings, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from qdrant_client import QdrantClient
import os
from pathlib import Path
import pandas as pd
import re
from bs4 import BeautifulSoup
import gc

# =========================
# CONFIG
# =========================
QDRANT_URL = os.environ.get("QDRANT_URL", "")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")


def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 1. DOCUMENTS
# =========================
print("Chargement des documents...")
documents = SimpleDirectoryReader(
    "text_dir/",
    file_metadata=add_file_metadata
).load_data()

# documents = documents[:10]
print(f"Documents chargés: {len(documents)}")


# =========================
# 2. CHUNKING
# =========================
print("Chunking...")
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

nodes = splitter.get_nodes_from_documents(documents)
print(f"Nombre de chunks: {len(nodes)}")

# =========================
# 3. EMBEDDINGS
# =========================
print("Chargement du modèle d'embedding...")
model_bge = "BAAI/bge-m3"

try:
    embed_model = HuggingFaceEmbedding(
        model_name=model_bge,
        device="cuda",
        max_length=1024,
        model_kwargs={"torch_dtype": "auto"}   # Limiter la longueur
    )
    Settings.embed_model = embed_model
    print("Modèle d'embedding chargé avec succès")
except Exception as e:
    print(f"Erreur lors du chargement du modèle: {e}")
    raise

# =========================
# 4. QDRANT
# =========================
print("Connexion à Qdrant...")
client = QdrantClient(path="./qdrant_local_bbai")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="legal_BAAI_bge-m3"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# =========================
# 5. INDEXATION DIRECTE
# =========================
print("Indexation...")
try:
    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        show_progress=True,
        embed_batch_size=2  # Afficher la progression
    )
    index.storage_context.persist(persist_dir="index_storage_bbai")
    print("Indexation complète OK.")
except Exception as e:
    print(f"Erreur lors de l'indexation: {e}")
    raise

c:\Users\alaa-\miniconda3\envs\tekno\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


Chargement des documents...
Documents chargés: 2634
Chunking...


2025-12-01 00:40:53,551 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Nombre de chunks: 29135
Chargement du modèle d'embedding...


`torch_dtype` is deprecated! Use `dtype` instead!


...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/463 [00:00<?, ?it/s]

Indexation complète OK.


In [5]:
print(documents[0])

Doc ID: af6d53c8-92a0-4856-ba71-43afce28c5c8
Text: All official European Union website addresses are in the
europa.eu domain. This document is an excerpt from the EUR-Lex website
Document 32024H1035 Commission Recommendation (EU) 2024/1035 of 23
February 2024 on the draft updated integrated national energy and
climate plan of Latvia covering the period 2021-2030 Commission
Recommendation (EU) 20...


In [ ]:
from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from qdrant_client import QdrantClient
import os
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================
QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]

# Choix du modèle d'embedding
# EMBED_MODEL_NAME = "BAAI/bge-m3"
# EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_MODEL_NAME = "BAAI/bge-m3"   # ← tu peux changer ici


def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 1. DOCUMENTS
# =========================
documents = SimpleDirectoryReader(
    "xml_dir/",
    file_metadata=add_file_metadata
).load_data()

print("len documents", len(documents))


# =========================
# 2. CHUNKING
# =========================
splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=100
)

nodes = splitter.get_nodes_from_documents(documents)


# =========================
# 3. EMBEDDINGS
# =========================
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.embed_model = embed_model


# =========================
# 4. QDRANT
# =========================
client = QdrantClient(path="./qdrant_local_bbai")

vector_store = QdrantVectorStore(
    client=client,
    collection_name="legal_" + EMBED_MODEL_NAME.replace("/", "_")
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# =========================
# 5. INDEXATION DIRECTE
# =========================
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

index.storage_context.persist(persist_dir="index_storage_bbai")

print(f"Indexation complète OK avec le modèle : {EMBED_MODEL_NAME}")


In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, PointStruct
import os
from dotenv import load_dotenv

load_dotenv()

# -------------------------
# CLIENT LOCAL
# -------------------------
local = QdrantClient(path="./qdrant_local_bbai")
collection_name = "legal_BAAI_bge-m3"

# -------------------------
# CLIENT PROD
# -------------------------
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
prod = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

# -------------------------
# CRÉER LA COLLECTION EN PROD
# -------------------------
prod.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=1024,
        distance="Cosine"
    )
)

# -------------------------
# EXPORT LOCAL
# -------------------------
all_points = []
offset = None

while True:
    points, offset = local.scroll(
        collection_name=collection_name,
        limit=5000,
        offset=offset,
        with_vectors=True,
        with_payload=True
    )
    all_points.extend(points)
    if offset is None:
        break

print("Nombre total de points :", len(all_points))

# -------------------------
# IMPORT EN PROD (BATCHS)
# -------------------------

batch_size = 10
for i in range(0, len(all_points), batch_size):
    batch = all_points[i:i + batch_size]
    prod.upsert(
        collection_name=collection_name,
        points=[
            PointStruct(
                id=p.id,
                vector=p.vector,
                payload=p.payload
            )
            for p in batch
        ]
    )
    print(f"Batch {i // batch_size + 1} envoyé ({len(batch)} points)")

print("Migration terminée en batchs.")


C:\Users\alaa-\AppData\Local\Temp\ipykernel_8412\1726095429.py:11: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <legal_BAAI_bge-m3> contains 29135 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  local = QdrantClient(path="./qdrant_local_bbai")
C:\Users\alaa-\AppData\Local\Temp\ipykernel_8412\1726095429.py:24: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  prod.recreate_collection(


Nombre total de points : 29135
Batch 1 envoyé (10 points)
Batch 2 envoyé (10 points)
Batch 3 envoyé (10 points)
Batch 4 envoyé (10 points)
Batch 5 envoyé (10 points)
Batch 6 envoyé (10 points)
Batch 7 envoyé (10 points)
Batch 8 envoyé (10 points)
Batch 9 envoyé (10 points)
Batch 10 envoyé (10 points)
Batch 11 envoyé (10 points)
Batch 12 envoyé (10 points)
Batch 13 envoyé (10 points)
Batch 14 envoyé (10 points)
Batch 15 envoyé (10 points)
Batch 16 envoyé (10 points)
Batch 17 envoyé (10 points)
Batch 18 envoyé (10 points)
Batch 19 envoyé (10 points)
Batch 20 envoyé (10 points)
Batch 21 envoyé (10 points)
Batch 22 envoyé (10 points)
Batch 23 envoyé (10 points)
Batch 24 envoyé (10 points)
Batch 25 envoyé (10 points)
Batch 26 envoyé (10 points)
Batch 27 envoyé (10 points)
Batch 28 envoyé (10 points)
Batch 29 envoyé (10 points)
Batch 30 envoyé (10 points)
Batch 31 envoyé (10 points)
Batch 32 envoyé (10 points)
Batch 33 envoyé (10 points)
Batch 34 envoyé (10 points)
Batch 35 envoyé (10 points

In [4]:
q = "Quelle est la portée de la directive 2008/50/CE sur la qualité de l’air ambiant et quelles obligations impose-t-elle aux États membres ?"

In [21]:
from llama_index.core.prompts import PromptTemplate

qa_tmpl = PromptTemplate("""
Vous êtes un moteur juridique. Répondez STRICTEMENT d'après les extraits fournis.

Chaque extrait inclut :
- son texte
- ses métadonnées (titre, type d'acte, numéro, date, chapitre)

Utilisez activement ces métadonnées pour éviter toute confusion entre directives.

=== CONTEXTE ===
{context_str}

=== METADONNEES ===
{metadata_str}

=== QUESTION ===
{query_str}

Réponse détaillée et fidèle au texte :
""")


In [24]:
query_engine = index.as_query_engine(
    similarity_top_k=30,
    rerank_top_k=6,
    verbose=True,
    response_mode = "compact",
    text_qa_template=qa_tmpl
)
response = query_engine.query(q)

print(response)

> Refine context: <p>(4) Directive 2001/81/EC of the European Par...
La directive 2008/50/CE sur la qualité de l'air ambiant vise à établir des normes pour l'air ambiant et à garantir un air plus propre en Europe. Elle impose aux États membres des obligations telles que la mise en place de mesures pour réduire les émissions de polluants atmosphériques, la surveillance de la qualité de l'air, et la communication de rapports réguliers à la Commission européenne. Cette directive est essentielle pour protéger la santé humaine et l'environnement en réduisant les niveaux de pollution atmosphérique et en améliorant la qualité de l'air dans l'Union européenne.


In [23]:
query_engine = index.as_query_engine(
    similarity_top_k=30,
    rerank_top_k=6,
    verbose=True,
    response_mode="tree_summarize",
    text_qa_template=qa_tmpl

)
response = query_engine.query(q)

print(response)

2 text chunks after repacking
1 text chunks after repacking
The directive 2008/50/CE on ambient air quality establishes standards for assessing and managing air quality, and it imposes obligations on Member States such as setting emission limits for certain air pollutants, including nitrogen dioxide, particulate matter, and lead in ambient air. Member States are required to adopt legislative, regulatory, and administrative measures to comply with this directive and communicate the main provisions of their national legislation in this area to the Commission.


In [ ]:
print(dir(response))
print(response.__dict__)
import json

def safe(obj):
    try:
        json.dumps(obj)
        return obj
    except:
        return str(obj)

clean = {k: safe(v) for k, v in result.__dict__.items()}

with open("./result_1.json", "w", encoding="utf-8") as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'get_formatted_sources', 'metadata', 'response', 'source_nodes']
{'response': "Une loi importante dans le cadre de l'environnement en Europe est la Directive (UE) 2024/1203 du Parlement européen et du Conseil du 11 avril 2024 sur la protection de l'environnement par le droit pénal. Cette directive vise à établir des dispositions pénales pour protéger l'environnement en criminalisant certaines actions nuisibles à l'environnement, telles que la pollution, le traitement illégal des déchets, et l'introduction illégale d'énergie dans l'environnement.", 'source_nodes': 